# 01 — Data Pipeline Validator

Validates all API connections and shows data freshness.

In [ ]:
import json, os, sys
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))
print(f"Root: {ROOT}")
print(f"Data dir: {DATA}")


## API Connectivity Checks

In [ ]:
import urllib.request, urllib.error, time

def check_url(label, url, timeout=10):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "UI-Index-NB/1.0"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            print(f"  ✅ {label}: HTTP {r.status}")
    except urllib.error.HTTPError as e:
        print(f"  ⚠️  {label}: HTTP {e.code}")
    except Exception as e:
        print(f"  ❌ {label}: {e}")
    time.sleep(0.3)

print("Checking API endpoints...")
check_url("BLS API v2",       "https://api.bls.gov/publicAPI/v2/timeseries/data/")
check_url("BLS QCEW",         "https://data.bls.gov/cew/data/api/2023/a/area/24000/super_sector/10/industry/10/size/0/ownership/0.json")
check_url("Census ACS",       "https://api.census.gov/data/2022/acs/acs5?get=NAME&for=state:24")
check_url("FEC API",          "https://api.open.fec.gov/v1/candidates/search/?api_key=DEMO_KEY&state=MD&limit=1")
check_url("USASpending",      "https://api.usaspending.gov/api/v2/references/agency/016/")
check_url("HUD FMR",          "https://www.huduser.gov/hudapi/public/fmr/statedata/MD")
check_url("Congress.gov",     "https://api.congress.gov/v3/member?api_key=DEMO_KEY&limit=1&format=json")
check_url("DOL ETA downloads","https://oui.doleta.gov/unemploy/DataDownloads.asp")


## Data File Freshness

In [ ]:
files = {
    "dmv_macro_baselines.csv": DATA / "dmv_macro_baselines.csv",
    "cpi_annual.json":         DATA / "cpi_annual.json",
    "inflation_crosscheck.json": DATA / "inflation_crosscheck.json",
    "county_data.json":        DATA / "county_data.json",
    "sui_rates.json":          DATA / "sui_rates.json",
    "fec_funding_profiles.json": DATA / "political" / "fec_funding_profiles.json",
    "federal_spending.json":   DATA / "political" / "federal_spending.json",
}

from datetime import datetime
import os

print(f"{'File':<35} {'Exists':<8} {'Last Modified'}")
print("-" * 65)
for label, path in files.items():
    exists = path.exists()
    mtime = datetime.fromtimestamp(os.path.getmtime(path)).strftime("%Y-%m-%d %H:%M") if exists else "—"
    status = "✅" if exists else "❌"
    print(f"{status} {label:<33} {mtime}")


## BLS vs FRED Inflation Cross-Check

In [ ]:
crosscheck_path = DATA / "inflation_crosscheck.json"
if crosscheck_path.exists():
    with open(crosscheck_path) as f:
        data = json.load(f)
    print(f"Fetched: {data['_metadata'].get('fetched_at', 'unknown')}")
    for jur, info in data.get("crosscheck", {}).items():
        flag = info.get("flag", "?")
        delta = info.get("national_vs_metro_delta_pct")
        icon = "⚠️" if flag == "REVIEW" else "✅"
        print(f"  {icon} {jur}: BLS vs DC-metro delta = {f'{delta:+.2f}%' if delta else 'N/A'} [{flag}]")
else:
    print("⚠️  Run fetch_fred_inflation.py to enable cross-check")
